# Stock Market Sentiment Analysis Using Tweets

**Text Mining group project — NOVA IMS, 2024/2025**

**Contributors:** Ana Caleiro, André Lourenço, Duarte Marques, Érica Parracho, and Jude Gbenimako.

This notebook is a cleaned archival copy of the original academic work. Saved execution outputs have been removed; original training and test CSVs are not distributed in this portfolio repository. The code has not been re-executed or updated for current library compatibility.


## Previously recorded validation result

The final submission pipeline used **CountVectorizer → TF-IDF Transformer → Logistic Regression**, with random oversampling applied to the training partition. The original notebook reported **accuracy 0.80** and **macro F1 0.73** on its validation split. This is a separately run experiment with a different validation setup, so these scores are not directly interchangeable with the broader comparison results. The cleaned copy has not been re-executed.


# **Table of Contents**

>[0. Context and Metadata](#scrollTo=gMekE_73wJIn)

>[1. Importing Libraries and Loading Data](#scrollTo=hGeLbOOPwRkz)

>>[1.1. Importing Libraries](#scrollTo=dZtGYsHCwWQH)

>>[1.2. Functions](#scrollTo=DUNto9cfeY84)

>>[1.3. Loading and Reading the Data](#scrollTo=io22uqHZw2C0)

>>[2. Preprocessing](#scrollTo=G1ifcA0IzzQs)

>>[2.1. Cleaning the Data](#scrollTo=l0GHs5800ubH)

>>[2.2. More Data Exploration](#scrollTo=-KqC96Ak1XRw)

>>[2.3. More Preprocessing](#scrollTo=4j-yf3SQ1QWf)

>>>[2.3.1. BertTokenizerFas](#scrollTo=-lfw2h4u16vH)

>[3. Feature Engineering](#scrollTo=5F5NqSAyfzXS)

>>[3.1 Train and Evaluation Split](#scrollTo=fArtxRpK3cqE)

>>[3.2. Oversampling](#scrollTo=3uPRgvcf45IV)

>>[3.3. Bag-of-Words + TF-IDF Vectorization](#scrollTo=cNWKqfNt4zhk)

>[4. Modeling](#scrollTo=QHYRYMw23x46)

>[5. Prediction](#scrollTo=u16G9s-E37Mb)



# **0. Context and Metadata**

We are performing a sentiment analysis of the stock market, predicting its behavior based on tweets. To accomplish this, we will use a classification model and NLP techniques to speculate the sentiment of each tweet (Bearish (0), Bullish (1), or Neutral (2)).

**Purpose of the notebook:**

In this notebook, we showcase our final solution (final piepline and the our best classfication model).


# **1. Importing Libraries and Loading Data**


## **1.1. Importing Libraries**

In [ ]:
#General Packages
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

#Preprocessing
import re, string

#Classification
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import LogisticRegression

#transformers
from transformers import BertTokenizerFast

#keras
import tensorflow as tf
from tensorflow import keras

#metrics
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

#others
import random
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split

random.seed(42)


## **1.2. Functions**

Preprocessing (Cleaning the Data)

In [ ]:
#Function that will remove ascii characters, punctuations, url, mentions and new line characters
def clean_overall(text):
    text = text.replace('\r', '').replace('\n', ' ').replace('\n', ' ').lower()
    text = re.sub(r"(?:\@|https?\://)\S+", "", text)
    text = re.sub(r'[^\x00-\x7f]',r'', text) #remove non utf8/ascii characters such as '\x9a\x91\x97\'
    banned_list= string.punctuation + 'Ã'+'±'+'ã'+'¼'+'â'+'»'+'§'
    table = str.maketrans('', '', banned_list)
    text = text.translate(table)
    return text

#clean hashtags at the end of the sentence, and keep those in the middle of the sentence by removing just the # symbol
def remove_hashtags(tweet):
    new_tweet = " ".join(word.strip() for word in re.split('#(?!(?:hashtag)\b)[\w-]+(?=(?:\s+#[\w-]+)*\s*$)', tweet)) #remove last hashtags
    new_tweet2 = " ".join(word.strip() for word in re.split('#|_', new_tweet)) #remove hashtags symbol from words in the middle of the sentence
    return new_tweet2

#Filter special characters such as & and $ present in some words
def remove_filter_chars(text):
    clean = []
    for word in text.split(' '):
        if ('$' in word) | ('&' in word):
            clean.append('')
        else:
            clean.append(word)
    return ' '.join(clean)

def remove_mult_spaces(text):
    return re.sub("\s\s+" , " ", text)

## **1.3. Loading and Reading the Data**


In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
pd.set_option("display.max_columns", None)

In [ ]:
train.head()

## **2. Preprocessing**

## **2.1. Cleaning the Data**

In [ ]:
# For Training Data
new_texts = []
for text in train.text:
    new_texts.append(remove_mult_spaces(remove_filter_chars(remove_hashtags(clean_overall(text)))))

In [ ]:
# For Test Data
new_texts_test = []
for text in test.text:
    new_texts_test.append(remove_mult_spaces(remove_filter_chars(remove_hashtags(clean_overall(text)))))

In [ ]:
train['clean_text'] = new_texts
test['clean_text'] = new_texts_test

In [ ]:
train['clean_text'].head(35)

## **2.2. More Data Exploration**

In [ ]:
length = []
for text in train.clean_text:
    tweet_len = len(text.split())
    length.append(tweet_len)

train['length'] = length

In [ ]:
length_test = []
for text in test.clean_text:
    tweet_len = len(text.split())
    length_test.append(tweet_len)

test['length'] = length_test

After the cleaning, we want to check the amount of tweets that lost almost all the words

Training tweets with less than 10 words

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot(x='length', data=train[train['length']<10], palette='Blues')
plt.title('Training tweets with less than 10 words')
plt.yticks()
plt.ylabel('count')
plt.xlabel('')
plt.show()

Test tweets with less than 10 words

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot(x='length', data=test[test['length']<10], palette='Blues')
plt.title('Test tweets with less than 10 words')
plt.yticks()
plt.ylabel('count')
plt.xlabel('')
plt.show()

After the cleaning, we can see that we have some tweets (on the training set) that have no words left.
Due to this, we will remove those, and also the ones with less than 4 words, given they represent a very small number of observations

In [ ]:
print(f" Train Shape: {train.shape}")
print(f" Test Shape: {test.shape}")

In [ ]:
train = train[train['length'] > 3]

In [ ]:
print(f" Train Shape: {train.shape}")
print(f" Test Shape: {test.shape}")

## **2.3. More Preprocessing**

### 2.3.1. BertTokenizerFas

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

In [ ]:
token_lens = []

for i,txt in enumerate(train['clean_text'].values):
    tokens = tokenizer.encode(txt, max_length=512, truncation=True)
    token_lens.append(len(tokens))

In [ ]:
train['token_lens'] = token_lens


In [ ]:
train = train.sort_values(by='token_lens', ascending=False)
train.head(20)

Since the first 5 lines are clearly not english, we will remove them to reduce the noise

In [ ]:
train = train.iloc[5:]
train.head()

In [ ]:
train = train.sample(frac=1, random_state=0).reset_index(drop=True)

In [ ]:
token_lens_test = []

for i,txt in enumerate(test['clean_text'].values):
    tokens = tokenizer.encode(txt, max_length=512, truncation=True)
    token_lens_test.append(len(tokens))

In [ ]:
test['token_lens'] = token_lens_test

In [ ]:
test_ordered = test.sort_values(by='token_lens', ascending=False)
test_ordered.head(20)

In [ ]:
test

# **3. Feature Engineering**

## **3.1 Train and Evaluation Split**

In [ ]:
X = train['clean_text'].values
y = train['label'].values

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)

## **3.2. Oversampling**

In [ ]:
#Oversample only the training data
ros = RandomOverSampler(random_state=42)
X_train_os, y_train_os = ros.fit_resample(X_train.reshape(-1, 1), y_train)


In [ ]:
# Flatten the X array back to 1D
X_train_os = X_train_os.ravel()

In [ ]:
y_train_copy = y_train_os.copy()
y_valid_copy = y_valid.copy()

## **3.3. Bag-of-Words + TF-IDF Vectorization**

In [ ]:
test = test['text'].tolist()

In [ ]:
clf = CountVectorizer()
X_train_cv =  clf.fit_transform(X_train_os)
X_valid_cv = clf.transform(X_valid)
test_cv = clf.transform(test)

In [ ]:
tf_transformer = TfidfTransformer(use_idf=True).fit(X_train_cv)
X_train_tf = tf_transformer.transform(X_train_cv)
X_valid_tf = tf_transformer.transform(X_valid_cv)
test_tf = tf_transformer.transform(test_cv)

# **4. Modeling**

In [ ]:
model_LR = LogisticRegression(solver='lbfgs', max_iter=500)
model_LR.fit(X_train_tf, y_train_copy)
train_pred_lr = model_LR.predict(X_train_tf)
lr_pred = model_LR.predict(X_valid_tf)

print('\tTrain Classification Report for Logistic Regression:\n\n',classification_report(y_train_copy,train_pred_lr))
print('\tValidation Classification Report for Logistic Regression:\n\n',classification_report(y_valid_copy,lr_pred))

In [ ]:
# Compute the confusion matrix
cm = confusion_matrix(y_valid_copy, lr_pred)

# Display it
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0, 1, 2], yticklabels=[0, 1, 2])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Logistic Regression')
plt.show()

# **5. Prediction**

In [ ]:
y_pred = model_LR.predict(test_tf)

# Get original y for submission

submission = pd.DataFrame({
    "text": test,
    "label": y_pred
})


# # Save to CSV in the required format
submission.to_csv("pred_23.csv", index=False)